# Baseline LLM-as-a-Judge Experiment

**How to use:** Edit the CONFIG cell below, then click **Run All**.  
This notebook contains no logic — all logic lives in `baseline_utils.py`.

---
## CONFIG — only edit this cell

In [ ]:
CONFIG_YAML  = "config.yaml"   # path to config file (do not change)
SLICE_N      = 3               # small number to test; set to None for full dataset
N_BOOTSTRAP  = 500             # 500 for testing; 2000 for the final thesis run
RESULTS_DIR  = "saved_results"

---
## Imports

In [ ]:
import os, sys, pickle
from pathlib import Path
import pandas as pd

# Find the repo root so Python can find LLM_models_interface
_root = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(_root, "LLM_models_interface")):
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

from LLM_models_interface.llm_interface import LLMJudge, load_configs, load_dataset
from baseline_utils import (
    load_from_cache, save_to_cache,
    compute_summary_row, build_decision_table, plot_baseline_figure,
)

# Create output folders
results_dir = Path(RESULTS_DIR)
cache_dir   = results_dir / "cache"
cache_dir.mkdir(parents=True, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/checkpoints", exist_ok=True)

print("Imports OK")
print(f"Results will be saved to: {results_dir.resolve()}")

---
## Load config and dataset

In [ ]:
configs = load_configs(CONFIG_YAML)

# Override slice_n from the CONFIG cell
if SLICE_N is not None:
    for cfg in configs:
        cfg.slice_n = SLICE_N

print(f"{len(configs)} experiment(s) loaded:")
for cfg in configs:
    print(f"  {cfg.name}  |  model={cfg.model}  |  backend={cfg.backend}  |  shots={cfg.shots}  |  slice_n={cfg.slice_n}")

---
## Run the judge loop

For each experiment and each trace:
- If a result is already cached → load it (no API call, no cost)
- Otherwise → call the model and save the result immediately

If this cell crashes, just run it again — completed traces are already saved.

In [ ]:
all_results = {}  # experiment name -> list of JudgeResponse
all_traces  = {}  # experiment name -> list of trace dicts (needed for ground truth)

for cfg in configs:
    print(f"\n{'='*60}")
    print(f"Experiment : {cfg.name}")
    print(f"Model      : {cfg.model}  |  backend: {cfg.backend}")
    print(f"{'='*60}")

    judge  = LLMJudge(cfg)
    traces = load_dataset(cfg)

    # Keep only Round 3 if the dataset has rounds
    if traces and "round" in traces[0]:
        traces = [t for t in traces if t.get("round") == "Round 3"]

    results  = []
    n_cached = 0
    n_called = 0
    checkpoint_path = f"{RESULTS_DIR}/checkpoints/{cfg.name}.pkl"

    for i, record in enumerate(traces):
        is_full    = isinstance(record["trace"], dict)
        trace_id   = f"{record['trace']['key']}_{record['trace_id']}" if is_full else str(record["trace_id"])
        trace_text = record["trace"]["trajectory"] if is_full else record["trace"]

        # Truncate very long traces so they fit in the context window
        if len(trace_text) + len(judge.examples) > 1048570:
            trace_text = trace_text[:1048570 - len(judge.examples)]

        # Try loading from cache first
        cached = load_from_cache(cache_dir, cfg.model, trace_id)
        if cached is not None:
            results.append(cached)
            n_cached += 1
            print(f"  [{i+1:>3}/{len(traces)}] CACHED   trace_id={trace_id}")
            continue

        # Call the model
        try:
            response = judge.judge_trace(trace_id, trace_text)
            save_to_cache(cache_dir, cfg.model, trace_id, response)
            results.append(response)
            n_called += 1
            print(f"  [{i+1:>3}/{len(traces)}] OK       "
                  f"tokens=({response.tokens_in}in/{response.tokens_out}out)  "
                  f"latency={response.latency_s:.1f}s  "
                  f"cost=${response.cost_usd:.5f}")

            # Save checkpoint after every trace
            with open(checkpoint_path, "wb") as f:
                pickle.dump(results, f)

            # Extra backup every 10 traces
            if len(results) % 10 == 0:
                with open(f"{RESULTS_DIR}/checkpoints/{cfg.name}_backup_{len(results)}.pkl", "wb") as f:
                    pickle.dump(results, f)

        except Exception as e:
            print(f"  [{i+1:>3}/{len(traces)}] ERROR    trace_id={trace_id}: {e}")
            with open(checkpoint_path, "wb") as f:
                pickle.dump(results, f)

    all_results[cfg.name] = results
    all_traces[cfg.name]  = traces
    print(f"\nDone: {n_called} new API calls, {n_cached} from cache, {len(results)} total.")

print("\nAll experiments complete.")

---
## Compute metrics

In [ ]:
summary_rows    = []
per_mode_rows   = []
prediction_rows = []

for cfg in configs:
    results = all_results.get(cfg.name, [])
    traces  = all_traces.get(cfg.name, [])
    if not results:
        print(f"[{cfg.name}] No results — skipping.")
        continue

    print(f"[{cfg.name}] Computing metrics ({N_BOOTSTRAP} bootstrap resamples)...", end=" ")
    summary, per_mode, predictions = compute_summary_row(
        cfg.name, cfg.model, results, traces, n_bootstrap=N_BOOTSTRAP
    )
    summary_rows.append(summary)
    per_mode_rows.extend(per_mode)
    prediction_rows.extend(predictions)
    print(f"macro_F1={summary['macro_f1']:.3f}  kappa={summary['kappa']:.3f}  cost=${summary['total_cost_usd']:.4f}")

print("\nMetrics done.")

---
## Decision table

Rows = experiments, columns = per-mode F1, macro F1, micro F1, kappa, cost, latency.

In [ ]:
decision_table = build_decision_table(summary_rows, per_mode_rows)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.3f}".format)
display(decision_table)

---
## Save results

In [ ]:
pd.DataFrame(summary_rows).to_csv(f"{RESULTS_DIR}/summary.csv", index=False)
pd.DataFrame(per_mode_rows).to_csv(f"{RESULTS_DIR}/metrics_per_mode.csv", index=False)
pd.DataFrame(prediction_rows).to_csv(f"{RESULTS_DIR}/predictions.csv", index=False)
decision_table.to_csv(f"{RESULTS_DIR}/decision_table.csv")

print(f"Saved to {RESULTS_DIR}/")
print("  summary.csv          — one row per experiment")
print("  metrics_per_mode.csv — one row per (experiment, failure mode)")
print("  predictions.csv      — one row per (experiment, trace)")
print("  decision_table.csv   — the baseline table")

---
## Figure

In [ ]:
plot_baseline_figure(
    decision_table,
    save_path=f"{RESULTS_DIR}/baseline_figure.png",
)